# ThermalGuard project notebook

This executed notebook presents the reproducible evidence for the AI for Engineers project. The reusable implementation lives in `src/thermalguard`; this notebook reads the generated outputs instead of duplicating the training code.


## Problem and evaluation rule

The target is permanent-magnet rotor temperature (`pm`). Complete `profile_id` sessions are kept together across training, validation, and final test data. The test profiles are used once, after validation MAE selects the model.


In [1]:
from pathlib import Path
import json
import pandas as pd

cwd = Path.cwd().resolve()
ROOT = cwd.parent if cwd.name == 'notebooks' else cwd
RESULTS = ROOT / 'reports'
assert (ROOT / 'src/thermalguard').is_dir(), f'Project root not found from {cwd}'


Project root: C:\Users\Kanha Gupta\Downloads\AI-Project


## Dataset validation


In [2]:
dataset = json.loads((RESULTS / 'metrics/dataset_summary.json').read_text())
print(f"rows={dataset['dataset']['rows']:,}")
print(f"columns={dataset['dataset']['columns']}")
print(f"profiles={dataset['dataset']['profiles']}")
print(f"missing_values={dataset['dataset']['missing_values']}")
print(f"duplicate_rows={dataset['dataset']['duplicate_rows']}")
print(f"target_range={dataset['target']['minimum_celsius']:.2f} to {dataset['target']['maximum_celsius']:.2f} C")


rows=1,330,816
columns=13
profiles=69
missing_values=0
duplicate_rows=0
target_range=20.86 to 113.61 C


![Dataset overview](../reports/figures/dataset_overview.png)


## Feature configurations

The controller set uses ambient and coolant temperature, speed, torque, d-q voltage, d-q current, current magnitude, voltage magnitude, and coolant minus ambient temperature. The enhanced set adds stator yoke, stator tooth, and stator winding temperatures.


![Correlation heatmap](../reports/figures/correlation_heatmap.png)


## Validation model comparison


In [3]:
comparison = json.loads((RESULTS / 'metrics/model_comparison.json').read_text())
rows = []
for name, metrics in comparison['validation_models'].items():
    rows.append({'feature_model': name, 'MAE_C': metrics['mae_celsius'], 'RMSE_C': metrics['rmse_celsius'], 'R2': metrics['r2']})
pd.DataFrame(rows).round(3)


feature_model | MAE_C | RMSE_C | R2
controller__mean_baseline | 15.679 | 19.186 | -0.013
controller__ridge | 10.457 | 13.028 | 0.533
controller__hist_gradient_boosting | 7.752 | 9.952 | 0.727
controller__xgboost | 8.113 | 10.573 | 0.692
enhanced__mean_baseline | 15.679 | 19.186 | -0.013
enhanced__ridge | 5.849 | 7.095 | 0.861
enhanced__hist_gradient_boosting | 4.610 | 6.115 | 0.897
enhanced__xgboost | 4.806 | 6.419 | 0.887


The enhanced Histogram Gradient Boosting model has the lowest validation MAE and is selected before the final test is evaluated.


## Final test result


In [4]:
test = comparison['test_metrics']
print(f"selected_model={comparison['selected_model']}")
print(f"MAE={test['mae_celsius']:.4f} C")
print(f"RMSE={test['rmse_celsius']:.4f} C")
print(f"R2={test['r2']:.4f}")
print(f"within_5C={test['within_5c_percent']:.2f}%")
print(f"bias={test['bias_celsius']:.4f} C")


selected_model=enhanced__hist_gradient_boosting
MAE=5.5969 C
RMSE=7.3605 C
R2=0.8557
within_5C=54.36%
bias=-0.3433 C


![Final model diagnostics](../reports/figures/final_model_diagnostics.png)


## Profile-level error analysis


In [5]:
profiles = pd.read_csv(RESULTS / 'tables/profile_metrics.csv')
profiles.head()


profile_id,rows,mae_celsius,rmse_celsius,r2,within_5c_percent,bias_celsius
45,17142,8.896418221287865,10.890221571408318,0.4288254759455411,33.309998833274996,-2.2621878827547035
59,7475,8.291295330828476,9.706822020074892,0.22922037384855953,28.160535117056856,-2.9714283341533068
43,8443,8.148878929572263,9.624422644892437,0.1340323903870403,35.27182281179675,2.4690657544915133
11,7887,7.366891386990681,10.830894202332376,-1.2358152897127237,51.020666920248516,5.4166726624420205
71,14656,7.275740771942285,9.619369074106418,0.49413566210151383,48.20551310043668,-2.7070960774468635


## Permutation importance


In [6]:
importance = pd.read_csv(RESULTS / 'tables/permutation_importance.csv')
importance.head()


feature,mae_increase_celsius,importance_std_celsius
stator_tooth,13.25089418225185,0.04299979189819491
stator_yoke,7.219048984363219,0.05353892402235593
stator_winding,2.3041540811751053,0.0038936923832150272
coolant_ambient_delta,1.9264667769590735,0.02561145644057712
ambient,1.4559732919549662,0.01288391298429442


Stator temperature sensors have the largest measured predictive influence. Permutation importance is not a causal claim, especially when thermal sensors are correlated.


## Saved artifact prediction


In [7]:
from thermalguard.artifact import load_artifact, predict_frame
from thermalguard.app_logic import build_input_frame

artifact = load_artifact(ROOT / 'models/thermalguard_model.joblib')
values = {name: limits['median'] for name, limits in artifact['input_summary'].items()}
frame = build_input_frame(values, artifact['input_summary'])
prediction = float(predict_frame(artifact, frame)[0])
print(f"Median-input prediction: {prediction:.2f} C")


Median-input prediction: 59.52 C


## Conclusion

The selected model reaches 5.60 C MAE and R2 of 0.856 on 14 unseen profiles. The result supports the use of nearby stator temperatures, but profile-level errors and the missing thermal history prevent this educational prototype from being treated as a certified control or protection system.
